In [1]:
#定义奖励模型
import torch
from typing import Optional
from torch import nn
import numpy as np
import os
from sympy import false
from transformers import AutoModelForCausalLM, AutoTokenizer

class RewardHead(nn.Module):
    """
    RewardHead类给GPT2实现了一个“头”，为每个输出的token返回一个标量值。
    """

    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.reward = nn.Linear(self.hidden_size, 1)
        self._post_init()

    def _post_init(self):
        nn.init.normal_(
            self.reward.weight,
            std=(1.0 / np.sqrt(self.hidden_size + 1))
        )
        nn.init.zeros_(self.reward.bias)

    def forward(self, hidden_states):
        output = hidden_states
        return self.reward(output)


class GPT2RewardModel(nn.Module):
    """
    GPT2模型加上一个“奖励头”
    """

    def __init__(self, model_name):
        super().__init__()
        self.llm = AutoModelForCausalLM.from_pretrained(model_name)
        # 添加奖励头
        self.reward_head = RewardHead(self.llm.config)

    def forward(
        self,
        input_ids,
        attention_mask,
    ) -> Optional[torch.FloatTensor]:
        # GPT2的输出
        transformer_outputs = self.llm.forward(
            input_ids,
            attention_mask=attention_mask,
            output_hidden_states = True,
        )

        # 获取最后一层隐藏层
        last_hidden_state = transformer_outputs.hidden_states[-1]

        # 对隐藏层给出奖励
        rewards = self.reward_head(last_hidden_state).squeeze(-1)
        # 归一化
        return torch.sigmoid(rewards)

#加载模型
model_path = "C:\\Users\lihaodong\.cache\modelscope\hub\models\Fengshenbang\Wenzhong-GPT2-110M-chinese-v2"
reward_model = GPT2RewardModel(model_path)
if os.path.exists('models/best_reward_model.pt'):
    reward_model.load_state_dict(torch.load('models/best_reward_model.pt'))
    print('reward_model.pt loaded')



D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.regi

In [2]:
#将最开始的SFT和价值模型训练代码放在一起，价值模型就是SFT后面训练一个价值头，和奖励模型一样，价值模型就是评论家，奖励模型就是裁判。SFT训练好后，价值模型和奖励模型都要保存下来，以便后续PPO训练使用。
import torch
from typing import Optional
from torch import nn
import numpy as np
from transformers import AutoModelForCausalLM

class ValueHead(nn.Module):
    """
    ValueHead类为GPT2实现了一个“头”，会为输出的每个token返回一个标量值
    标量值就是这个token的价值，ValueHead就是评论家。
    """
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.value = nn.Linear(self.hidden_size, 1)
        self._post_init()

    def _post_init(self):
        nn.init.normal_(
            self.value.weight,
            std=(1.0 / np.sqrt(self.hidden_size + 1))
        )
        nn.init.zeros_(self.value.bias)

    def forward(self, hidden_states):
        output = hidden_states
        return self.value(output)

In [3]:
#整体模型
class ModelForCausalLMWithValueHead(nn.Module):
    """
    GPT2模型+一个价值头
    """
    def __init__(self, model_path):
        super().__init__()
        # 这个要初始化为我们微调出来的gpt2-sft模型
        # actor演员模型
        self.llm = AutoModelForCausalLM.from_pretrained(model_path)
        # 添加价值头
        # critic评论家模型
        self.v_head = ValueHead(self.llm.config)

    def forward(
        self,
        input_ids,
        attention_mask,
    ) -> Optional[torch.FloatTensor]:
        # gpt2-sft模型的输出
        transformer_outputs = self.llm.forward(
            input_ids,
            attention_mask=attention_mask,
            output_hidden_states = True,
        )
        # 输出的token的概率分布，维度为 `vocab_size`
        lm_logits = transformer_outputs.logits
        # 获取最后一层隐藏层
        last_hidden_state = transformer_outputs.hidden_states[-1]

        # 评估token的价值
        value = self.v_head(last_hidden_state).squeeze(-1)
        # 返回输出的token的logits和token的价值
        return lm_logits, value

    def generate(self, *args, **kwargs):
        return self.llm.generate(*args, **kwargs)

In [4]:
#初始化模型，加载微调好的gpt2-sft模型，后续PPO训练的时候就会在这个基础上继续训练价值头。
model_path = '../SFT微调大模型/models/fine_tuned_gpt2'
model = ModelForCausalLMWithValueHead(model_path)
# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(model_path)
# model.eval()
# prompt = "今天天气真好，我想去"   # 可以修改成你想要的任何开头
# inputs = tokenizer(prompt, return_tensors='pt')
# input_ids = inputs['input_ids']
# attention_mask = inputs['attention_mask']
#
# with torch.no_grad():
#     # 设置生成参数
#     outputs = model.generate(
#         input_ids=input_ids,
#         attention_mask=attention_mask,
#         max_new_tokens=100,          # 生成的新 token 数量，你可以调大
#         do_sample=True,              # 启用采样（随机性）
#         temperature=0.01,             # 控制随机性，值越小越保守
#         top_p=0.9,                   # nucleus sampling
#         repetition_penalty=1.1,      # 避免重复
#         pad_token_id=tokenizer.pad_token_id,
#         eos_token_id=tokenizer.eos_token_id,
#     )
#
# generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
# print("生成的文本：")
# print(generated_text)

In [5]:
import random
#准备数据集
from functools import partial
#加载tokenizer
from datasets import load_dataset
from transformers import AutoTokenizer


#定义collate
def reward_collate_fn(batch):
    # batch 是一个列表，每个元素是字典，包含 'input_ids', 'attention_mask', 'score', 'score_index' 等
    input_ids = [item['input_ids'] for item in batch]
    attention_mask = [item['attention_mask'] for item in batch]
    scores = [item['score'] for item in batch]
    # 原有的 score_index 不再使用，直接重新计算：
    # 每个样本的 reward token 在原始序列中的位置就是最后一个 token
    # 但经过 padding 后，我们需要找到每个样本的最后一个有效 token 的位置
    # 方法：先 padding，然后根据 attention_mask 找到每个样本的最后一个 1 的位置
    # 注意：padding 会改变序列长度，但我们可以在 padding 后再计算。

    # 使用 tokenizer 进行 padding
    padded = tokenizer.pad(
        {'input_ids': input_ids,
         'attention_mask': attention_mask},
        padding=True,
        return_tensors='pt'
    )

    # 计算每个样本的最后一个有效 token 的索引
    # attention_mask 是 (batch_size, seq_len)，每行是 0/1
    # 使用 torch.argmax 可以找到最后一个 1 的位置（因为 argmax 返回第一个最大值的索引，对于 0/1 序列，最后一个 1 的位置需要更复杂的方法）
    # 简便方法：取每行非零位置的最大索引
    seq_len = padded['attention_mask'].shape[1]
    # 创建一个索引矩阵
    indices = torch.arange(seq_len).unsqueeze(0).expand(padded['attention_mask'].shape[0], -1).to(padded['attention_mask'].device)
    # 将 attention_mask 为 0 的位置索引设为 -1
    masked_indices = indices * padded['attention_mask']
    # 取每行的最大值，就是最后一个有效 token 的索引
    score_index = masked_indices.max(dim=1).values

    # 将 scores 转换为 tensor
    scores_tensor = torch.tensor(scores, dtype=torch.float32)

    # 返回一个字典，包含 input_ids, attention_mask, score, score_index
    return {
        'input_ids': padded['input_ids'],
        'attention_mask': padded['attention_mask'],
        'score': scores_tensor,
        'score_index': score_index
    }

# 定义分词函数：接收一个 batch 字典，返回一个字典（键是新的列名，值是列表）
def tokenize(batch, tokenizer,input_token_length_range):
    input_sizes = [random.choice(input_token_length_range) for _ in range(len(batch['context']))]
    input_ids_list = []
    attention_mask_list = []
    query_list = []
    for context, input_size in zip(batch['context'], input_sizes):
        ids = tokenizer.encode(context)[:input_size]
        input_ids_list.append(ids)
        attention_mask_list.append([1] * len(ids))
        query_list.append(tokenizer.decode(ids))
    batch['input_ids'] = input_ids_list
    batch['attention_mask'] = attention_mask_list
    batch['query'] = query_list
    return batch


#清理数据长度国小的
def filter_short_samples(batch, min_length=8):
    # 计算输入 ID 的长度
    input_ids_length = len(batch['context'])
    # 只有当长度大于或等于 min_length和小于等于100 时才保留样本
    return (input_ids_length >= min_length and input_ids_length <= 150)

def load_data(data_path,tokenizer):
    dataset = load_dataset('json', data_files={
        'train': data_path[0],
        'valid': data_path[1],
        'test': data_path[2]
    })
    #过滤掉长度过短的样本，避免训练时出现过多无效样本
    dataset["train"] = dataset["train"].filter(filter_short_samples)
    dataset["valid"] = dataset["valid"].filter(filter_short_samples)

    #控制输出数量
    import random
    input_min_token_length = 3
    input_max_token_length = 20
    input_token_length_range = list(range(
        input_min_token_length,
        input_max_token_length))
    print(input_token_length_range)
    print(random.choice(input_token_length_range))

    # 查看结构
    print("未编码样本",dataset['train'][0])
    # 输出：{'idx': 0, 'context': '如果 我 无聊 时 ...', 'label': 1}

    map_kwargs = {
        'batched': True,  # 启用批处理模式，tokenize 接收的是 batch 字典
        'batch_size': 512,  # 每批 512 个样本
        'remove_columns': ['idx', 'context', 'label']  # 处理后移除这三列
    }

    # 在 map 时绑定 tokenizer
    tokenize_with_tokenizer = partial(tokenize, tokenizer=tokenizer,input_token_length_range=input_token_length_range)

    # 对训练集和验证集应用 map
    tokenized_dataset_train = dataset["train"].map(tokenize_with_tokenizer, **map_kwargs)
    tokenized_dataset_val = dataset["valid"].map(tokenize_with_tokenizer, **map_kwargs)

    print("训练样本",tokenized_dataset_train[0])
    print("测试样本",tokenized_dataset_val[0])

    tokenized_dataset_train.set_format(type='torch')
    tokenized_dataset_val.set_format(type='torch')

    print(tokenized_dataset_train[0])

    return tokenized_dataset_train, tokenized_dataset_val


data_path = ["../SFT微调大模型/data_raw/train.jsonl","../SFT微调大模型/data_raw/valid.jsonl","../SFT微调大模型/data_raw/test.jsonl"]
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenized_dataset_train, tokenized_dataset_val = load_data(data_path=data_path,tokenizer=tokenizer)

from torch.utils.data import DataLoader

batch_size = 32

def collator(batch):
    return dict((key, [d[key] for d in batch]) for key in batch[0])

train_dataloader = DataLoader(tokenized_dataset_train, batch_size=batch_size, collate_fn=collator, shuffle=True)
val_dataloader = DataLoader(tokenized_dataset_val, batch_size=batch_size, collate_fn=collator, shuffle=True)

batch = next(iter(train_dataloader))
print("数据：",batch,'\n')
print(batch.keys())



[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
7
未编码样本 {'idx': 0, 'context': '死囚爱刽子手女贼爱衙役我们爱你们难道还有别的选择没想到胡军除了蓝宇还有东宫西宫我个去阿兰这样真他nia恶心爱个P分明只是欲', 'label': 1}
训练样本 {'input_ids': [29826, 119, 32368, 248, 163, 230, 109, 26344, 121, 36310, 33699, 233, 42637, 164, 112, 120, 163, 230, 109], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'query': '死囚爱刽子手女贼爱'}
测试样本 {'input_ids': [20998, 108, 162], 'attention_mask': [1, 1, 1], 'query': '台�'}
{'input_ids': tensor([29826,   119, 32368,   248,   163,   230,   109, 26344,   121, 36310,
        33699,   233, 42637,   164,   112,   120,   163,   230,   109]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]), 'query': '死囚爱刽子手女贼爱'}
数据： {'input_ids': [tensor([42637, 17312,   233, 20998,   233, 17358,   223,   162,   109]), tensor([33176,   120,   163,   101,   248, 20998,   107, 20015]), tensor([33768,   254,   164,   223,   232, 45298, 43291, 43380]), tensor([22887,   109, 4246

In [6]:
output_min_length = 5
output_max_length = 20
# gpt2-sft输出的配置
# - 模型会从整个词汇表中按照原始概率分布进行采样
# - 每个词被选中的概率完全由模型的原始输出决定
generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0, # 所有词汇表中的词都可能被选中
    "top_p": 1.0, # 包含整个概率分布
    "do_sample": True,# 启用采样（随机性）
    "pad_token_id": tokenizer.pad_token_id # 填充标记的 ID，确保生成过程中正确处理填充
}
new_tokens = random.choice(list(range(output_min_length, output_max_length)))# 随机选择生成的 token 数量
generation_kwargs["max_new_tokens"] = new_tokens
sample = tokenizer('这是一个')
print(sample)
print(tokenizer.decode(sample['input_ids']))
#模型输出
query_response = model.generate(
    input_ids=torch.tensor(sample['input_ids']).unsqueeze(0),# 将输入 ID 转换为 tensor，并添加一个批次维度（batch_size=1）
    attention_mask=torch.tensor(sample['attention_mask']).unsqueeze(0),# 将注意力掩码转换为 tensor，并添加一个批次维度（batch_size=1）
    **generation_kwargs
).squeeze(0)
print(query_response)
print(tokenizer.decode(query_response))

{'input_ids': [32573, 247, 42468, 31660, 10310, 103], 'attention_mask': [1, 1, 1, 1, 1, 1]}
这是一个
tensor([32573,   247, 42468, 31660, 10310,   103, 49035,   226,   163,   122,
          236, 33176,   112, 47987,   163,   120,   241,   163,   120,   241,
        26193,   234,   164,   113])
这是一个凄美年代缓缓行�


In [7]:
REWARD_TOKEN_ID = tokenizer.eos_token_id
with torch.no_grad():
    query_response_score = torch.cat([
        query_response,
        torch.tensor([REWARD_TOKEN_ID])])# 在生成的文本后面添加一个特殊的奖励 token，reward_model 会根据这个 token 的位置来计算奖励
    attention_mask = torch.ones_like(query_response_score, dtype=torch.long)# 生成的文本和奖励 token 都是有效的输入，所以 attention_mask 全部设置为 1
    score = reward_model(
        query_response_score.unsqueeze(0),
        attention_mask.unsqueeze(0)
    ).squeeze(0)[-1]# reward_model 的输出是一个序列，每个 token 都有一个奖励值，我们取最后一个 token 的奖励值作为整个生成文本的奖励，因为我们在生成文本后面添加了一个特殊的奖励 token，reward_model 会根据这个 token 的位置来计算奖励，所以最后一个 token 的奖励值就是我们想要的奖励。
print(score)

tensor(0.1695)


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
reward_model = reward_model.to(device)

query_tensors = batch['input_ids']
query_attention_masks = batch['attention_mask']

response_tensors = []
query_response_tensors = []
score_tensors = []

for i, query in enumerate(query_tensors):
    query = query.to(device)
    query_attention_mask = query_attention_masks[i].to(device)
    new_tokens = random.choice(list(range(output_min_length, output_max_length)))
    generation_kwargs["max_new_tokens"] = new_tokens
    query_response = model.generate(
        input_ids=query.unsqueeze(0),
        attention_mask=query_attention_mask.unsqueeze(0),
        **generation_kwargs
    ).squeeze(0)

    response_len = len(query_response) - len(query)  # 生成的文本长度
    response_tensors.append(query_response[-response_len:]) # 只保留生成的文本部分
    query_response_tensors.append(query_response)# 将输入和生成的文本拼接在一起，作为奖励模型的输入

    with torch.no_grad():
        query_response_score = torch.cat([query_response, torch.tensor([REWARD_TOKEN_ID]).to(device)])# 在生成的文本后面添加一个特殊的奖励 token，reward_model 会根据这个 token 的位置来计算奖励
        attention_mask = torch.ones_like(query_response_score, dtype=torch.long)# 生成的文本和奖励 token 都是有效的输入，所以 attention_mask 全部设置为 1
        score = reward_model(
            query_response_score.unsqueeze(0),
            attention_mask.unsqueeze(0)
        ).squeeze(0)[-1]
        score = 2 * (score - 0.5)
    score_tensors.append(score)

batch["response"] = [tokenizer.decode(response) for response in response_tensors]# 将生成的文本转换为字符串，保存在 batch 中，方便后续查看
from pprint import pprint
pprint(batch['response'])
pprint(score_tensors)

['�很严格但是偏�',
 '�打分生命�',
 '�手法想着初�',
 '�满高尚爱情�',
 '证连瘾心中不亦�',
 '�og7也误导一些人不过',
 '�人脸蛋也太可�',
 '蔡康永幼�',
 '�看开',
 '�节莫',
 '提琴表演真',
 '��情励志悲剧�',
 '�捏逗灰太狼使�',
 '��只能说游戏开',
 '�磨�',
 '�时除了充',
 '是押鍒不速Mandy书序',
 '��斯丁结局一个',
 '��我一直以为小超人�',
 '看这种电影影�',
 '可以让粉丝远�',
 '估计最久会听�',
 '��编惊悚片段',
 '厘头没有演员没',
 'rique这种品叔从�',
 '�演手脚�',
 '�鹤立雀归隙当看x�',
 '看过NANA好喜',
 '�开心这部片子说�',
 '起跑公',
 '�注意小剧作�',
 '�为自由虚�']
[tensor(-0.6782, device='cuda:0'),
 tensor(-0.6767, device='cuda:0'),
 tensor(-0.6434, device='cuda:0'),
 tensor(-0.6535, device='cuda:0'),
 tensor(-0.6922, device='cuda:0'),
 tensor(-0.6967, device='cuda:0'),
 tensor(-0.6655, device='cuda:0'),
 tensor(-0.6866, device='cuda:0'),
 tensor(-0.6419, device='cuda:0'),
 tensor(-0.6649, device='cuda:0'),
 tensor(-0.6804, device='cuda:0'),
 tensor(-0.6515, device='cuda:0'),
 tensor(-0.6773, device='cuda:0'),
 tensor(-0.6690, device='cuda:0'),
 tensor(-0.6600, device='cuda:0'),
 tensor(-0.6634, device='cuda:0'),
 tensor(-0.6820, device='cuda:0'),
 tensor(-0.6767,

In [13]:
#冻结一个模型，用于后续的参数更新，使得后续模型不会偏离最开始的SFT模型太远，保持一定的稳定性。
from copy import deepcopy
sft_model = deepcopy(model)



$$ \begin{aligned} \text{reward} &= \text{score} - \log\left(\frac{\pi_\theta^{\text{RL}}}{\pi^{\text{SFT}}}\right) \ &= \text{score} - (\log\pi_\theta^{\text{RL}}-\log\pi^{\text{SFT}}) \end{aligned} $$

In [14]:
#数据处理
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

input_data = data_collator([
    {'input_ids': ids,
     'attention_mask': torch.ones_like(ids)} for ids in query_response_tensors
]).to(device)
print(input_data)


{'input_ids': tensor([[42637, 17312,   233,  ..., 50256, 50256, 50256],
        [33176,   120,   163,  ..., 50256, 50256, 50256],
        [33768,   254,   164,  ..., 50256, 50256, 50256],
        ...,
        [22755,   239,   162,  ..., 50256, 50256, 50256],
        [36181,   230,   161,  ..., 50256, 50256, 50256],
        [  163,   118,   107,  ..., 50256, 50256, 50256]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0')}


In [16]:
def compute_rewards(
    input_data,
    query_tensors,
    response_tensors,
    score_tensors # 是奖励模型对第 j 个样本整个生成文本给出的最终得分
):
    with torch.no_grad():
        # 正在微调的模型所输出的token的logits和token的价值
        # 模型输出所有token的概率分布
        logits, values = model(**input_data) # b, seq, vocab
        # 冻结的模型的输出和价值
        ref_logits, _ = sft_model(**input_data)
        # 正在微调的模型的输出的对数概率
        logp = torch.nn.functional.log_softmax(logits[:, :-1, :], dim=-1)#去掉最后一个token的logits，因为最后一个token是奖励token，不参与概率计算，然后对剩下的token的logits进行softmax沿着词汇表维度计算对数概率，得到每个token的log概率分布，维度为 batch_size, seq_len-1, vocab_size
        # 冻结的模型的输出的对数概率
        ref_logp = torch.nn.functional.log_softmax(ref_logits[:, :-1, :], dim=-1)
        # 实际生成的token序列
        labels = input_data['input_ids'][:, 1:] # b, seq  去掉第一个token，因为第一个token是输入的最后一个token，不参与概率计算，labels的维度为 batch_size, seq_len-1
        # 使用gather提取实际token的概率
        logp = torch.gather(
            logp,
            2,
            labels.unsqueeze(-1)
        ).squeeze(-1) # batch, seq  gather函数的作用是根据labels中的索引，从logp中提取对应位置的值，labels.unsqueeze(-1)将labels的维度从(batch_size, seq_len-1)变为(batch_size, seq_len-1, 1)，这样就可以在logp的词汇表维度上进行gather操作，提取出实际生成的token的log概率，最后squeeze(-1)将维度从(batch_size, seq_len-1, 1)变回(batch_size, seq_len-1)
        ref_logp = torch.gather(
            ref_logp,
            2,
            labels.unsqueeze(-1)
        ).squeeze(-1) # batch, seq
        # kl散度
        kl = logp - ref_logp
        # kl散度的权重
        beta = 0.2
        # 最终奖励的计算
        rewards = - beta * kl #形状是b ，seq-1  即每个生成动作的即时 KL 惩罚。
        attention_mask = input_data['attention_mask']
        masks = torch.zeros_like(attention_mask[:, 1:])  #形状 (seq_len-1,)，在生成部分的位置为 1，在 query 部分和 padding 部分为 0。
        masks[:,:] = attention_mask[:, 1:]
        for j in range(len(query_tensors)):
            start = len(query_tensors[j]) - 1 #start：生成部分的第一个 token 在预测序列中的索引。
            end = start + len(response_tensors[j]) #end：生成部分的结束索引（不包含），也就是reward token的索引。
            masks[j, :start] = 0# query部分的mask设置为0，不参与奖励计算，response部分的mask设置为1，参与奖励计算，reward token部分的mask设置为0，不参与奖励计算
            masks[j, end:] = 0
            rewards[j, end - 1] += score_tensors[j]  # 原本这个位置的奖励只有 KL 惩罚，加上 score_tensors[j] 后，该位置的总奖励变为 -β*kl + score。
            rewards[j, :] *= masks[j, :]# reward token部分的奖励已经加到response部分的最后一个token上了，所以reward token部分的mask设置为0，不参与奖励计算，response部分的mask设置为1，参与奖励计算，query部分的mask设置为0，不参与奖励计算，这样就可以在PPO训练时通过reward token的奖励来指导模型生成更好的文本，同时又不会让reward token直接参与概率计算，保持训练的稳定性。
            values[j, :-1] *= masks[j, :]

    return logp, rewards, values[:, :-1], masks

In [17]:
logprobs, rewards, values, masks = compute_rewards(
    input_data,
    query_tensors,
    response_tensors,
    score_tensors
)
print(rewards[0])
print(input_data['input_ids'][0])
print(input_data['attention_mask'][0])
print(masks[0])
print(values[0])

tensor([-0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000,
        -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000,
        -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.6782, -0.0000, -0.0000,
        -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000,
        -0.0000], device='cuda:0')
tensor([42637, 17312,   233, 20998,   233, 17358,   223,   162,   109,   224,
        36181,   230, 10310,    98, 43718,   120, 19526,   228, 42468,   161,
          223,   237,   161, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256], device='cuda:0')
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')
tensor([ 0.0000, -0.0000,  0.0000, -0.0000,  0.0000, -0.0000,  0.0000, -0.0000,
        

例子
## 一、模拟样本设定
我们通过**具体数值**模拟代码运行，清晰展示三行核心代码与循环的逻辑：
- **查询 tokens（query_tensors）**：`[1, 2, 3]`（长度 = 3）
- **回复 tokens（response_tensors）**：`[4, 5, 6]`（长度 = 3）
- **奖励标记 ID（reward token ID）**：`999`
- **最终得分（score_tensors[j]）**：`0.7`
- **模型完整输入序列（input_ids）**：query + response + reward token → `[1, 2, 3, 4, 5, 6, 999]`
- **序列总长度（seq_len）**：`7`
- **注意力掩码（attention_mask）**：全 `1`（所有位置有效）

---

## 二、初始化 masks 和 rewards
`rewards` 和 `masks` 长度为 `seq_len - 1 = 6`，对应**预测每个 token 的位置**：
1. **rewards 初始值**：每个位置为 KL 惩罚（负数）→ `[-0.1, -0.2, -0.3, -0.4, -0.5, -0.6]`
2. **masks 初始值**：由 `attention_mask[:, 1:]` 右移一位得到 → `[1, 1, 1, 1, 1, 1]`

---

## 三、循环内核心代码执行
### 核心代码
```python
for j in range(len(query_tensors)):
    # 1. 定位回复 token 的起始/结束索引
    start = len(query_tensors[j]) - 1   # 3 - 1 = 2
    end = start + len(response_tensors[j])  # 2 + 3 = 5

    # 2. 掩码清零：只保留回复生成部分
    masks[j, :start] = 0   # 索引 0、1 清零（query 部分）
    masks[j, end:] = 0     # 索引 5 清零（reward token 部分）

    # 3. 最终得分加到生成部分的最后一个 token
    rewards[j, end - 1] += score_tensors[j]   # end-1 = 4

    # 4. 掩码过滤奖励和价值，只保留生成部分
    rewards[j, :] *= masks[j, :]
    values[j, :-1] *= masks[j, :]
```

### 执行后结果
1. **masks 最终值**：`[0, 0, 1, 1, 1, 0]`
2. **rewards 变化**
   - 原始：`[-0.1, -0.2, -0.3, -0.4, -0.5, -0.6]`
   - 加分后：`[-0.1, -0.2, -0.3, -0.4, 0.2, -0.6]`
   - 掩码过滤后：`[0, 0, -0.3, -0.4, 0.2, 0]`
3. **values 变化**
   - 原始（前6位）：`[1.0, 2.0, 3.0, 4.0, 5.0, 6.0]`
   - 掩码过滤后：`[0, 0, 3.0, 4.0, 5.0, 0]`

---

## 四、执行结果可视化表格
| rewards/values 位置索引 | 对应 token       | 是否为生成部分 | 原始 rewards | 最终 rewards | 原始 values | 最终 values |
|------------------------|------------------|----------------|--------------|---------------|-------------|-------------|
| 0                      | query[1]         | 否             | -0.1         | 0             | 1.0         | 0           |
| 1                      | query[2]         | 否             | -0.2         | 0             | 2.0         | 0           |
| 2                      | response[0] (4)  | 是             | -0.3         | -0.3          | 3.0         | 3.0         |
| 3                      | response[1] (5)  | 是             | -0.4         | -0.4          | 4.0         | 4.0         |
| 4                      | response[2] (6)  | 是             | -0.5 → 0.2   | 0.2           | 5.0         | 5.0         |
| 5                      | reward token(999)| 否             | -0.6         | 0             | 6.0         | 0           |



In [23]:
def masked_mean(values, mask):
    # 计算带掩码的平均值
    return (values * mask).sum() / mask.sum()

def masked_var(values, mask):
    # 计算带掩码的方差
    mean = masked_mean(values, mask)
    centred_values = values - mean
    return masked_mean(centred_values ** 2, mask)

def masked_whiten(values, mask):
    '''
    对数据进行带掩码的白化处理，
    让有效数据的方差变为1，但均值保持不变
    '''
    mean, var = masked_mean(values, mask), masked_var(values, mask)
    whitened = (values - mean) * torch.rsqrt(var + 1e-8)
    whitened += mean
    return whitened

def compute_advantage(rewards, values, masks):
    '''
    广义优势估计（GAE）
    '''
    lastgae = 0.0 #
    advantage_reversed = []
    seq_length = rewards.shape[-1]
    gamma, lam = 1.0, 0.95

    for t in reversed(range(seq_length)):
        nextvalues = values[:, t + 1] if t < seq_length - 1 else 0.0 #从后往前计算优势，t+1是下一个时间步的价值，如果t已经是最后一个时间步了，那么下一个时间步的价值就是0
        delta = rewards[:, t] + gamma * nextvalues - values[:, t] #优势计算，当前步的奖励加上下一个时间步的价值折扣减去当前时间步的价值，得到当前时间步的优势
        lastgae = delta + gamma * lam * lastgae #这个就是GAE的核心公式，当前时间步的优势等于当前时间步的delta加上折扣后的下一个时间步的优势，gamma是折扣因子，lam是GAE的平滑参数
        advantage_reversed.append(lastgae) #将计算得到的优势添加到列表中，注意这里是从后往前计算的，所以最后需要反转列表
    advantages = torch.stack(advantage_reversed[::-1], dim=1)#将列表反转过来，得到正确的时间顺序，然后沿着时间维度堆叠成一个张量，维度为(batch_size, seq_len)
    advantages = masked_whiten(advantages, masks)

    returns = advantages + values
    return advantages, returns #returns 是目标回报，用于价值网络的回归损失。

advantages, returns = compute_advantage(rewards, values, masks)
print(advantages[0])
print(returns[0])

tensor([-0.2199, -0.2320, -0.2448, -0.2582, -0.2724, -0.2872, -0.3029, -0.3194,
         0.0024, -1.1377,  1.0982,  0.0058, -1.1463, -0.7757,  1.0522, -1.3234,
        -0.4651, -0.9069, -1.0018, -1.2631, -1.0300, -0.2872,  0.0105,  0.0105,
         0.0105,  0.0105,  0.0105,  0.0105,  0.0105,  0.0105,  0.0105,  0.0105,
         0.0105], device='cuda:0')
tensor([-0.2199, -0.2320, -0.2448, -0.2582, -0.2724, -0.2872, -0.3029, -0.3194,
        -0.3115, -0.3968, -0.2862, -0.3146, -0.4006, -0.4294, -0.3315, -0.4577,
        -0.4588, -0.5148, -0.5666, -0.6354, -0.6800, -0.6754,  0.0105,  0.0105,
         0.0105,  0.0105,  0.0105,  0.0105,  0.0105,  0.0105,  0.0105,  0.0105,
         0.0105], device='cuda:0')


In [24]:
learning_rate = 1e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
# 随机排列一下各个批次大小
np.random.permutation(batch_size)


array([11, 21, 29, 18,  6, 20, 22,  3,  1, 30,  7, 15, 12,  9,  8, 13, 14,
        4, 17,  5, 31, 27, 19, 28, 25, 10,  2, 26, 16,  0, 24, 23],
      dtype=int32)

In [26]:
# 最小的批次大小
mini_batch_size = 4
# 训练 4 个 epoch
ppo_epochs = 4
# ε = 0.2
cliprange_ratio = 0.2

v_loss_coeff = 0.1
# 比例的阈值
ratio_threshold = 10

def compute_loss(
    old_logprobs,
    values,
    logprobs,
    vpreds,
    masks,
    advantages,
    returns
):
    ratio = torch.exp(logprobs - old_logprobs)
    pg_loss1 = - ratio * advantages
    pg_loss2 = - torch.clamp(
        ratio,
        1 - cliprange_ratio,
        1 + cliprange_ratio
    ) * advantages
    pg_loss = masked_mean(torch.max(pg_loss1, pg_loss2), masks) #只计算生成部分的策略损失，取的是pg_loss1, pg_loss2的最大值，原始式子是取最小值的，但是因为pg_loss1和pg_loss2都是负数，所以取最大值相当于取最小值的绝对值

    v_loss = masked_mean((vpreds - returns) ** 2, masks)#只计算生成部分的价值损失，vpreds是模型预测的价值，returns是目标回报，两者的差的平方就是价值损失，masked_mean函数会根据masks来计算平均值，只考虑生成部分的损失。
    loss = pg_loss + v_loss_coeff * v_loss

    #如果平均概率比超过阈值，说明新策略与旧策略差异太大，可能会造成训练不稳定，因此将损失置零，跳过本次更新。
    avg_ratio = masked_mean(ratio, masks)
    if avg_ratio > ratio_threshold:
        pg_loss = pg_loss * 0.0
        v_loss = v_loss * 0.0
        loss = loss * 0.0

    return loss, v_loss

def mini_batch_train():
    # 过滤掉输入数据为空的批次
    if input_data['input_ids'].shape[0] == 0:
        return
    for ep in range(ppo_epochs):
        batch_inds = np.random.permutation(batch_size) # 随机打乱批次索引，确保每个 epoch 的训练顺序不同，有助于模型更好地泛化。

        for start in range(0, batch_size, mini_batch_size):
            end = start + mini_batch_size
            mini_batch_inds = batch_inds[start:end]

            mb_model_inputs = {
                'input_ids': input_data['input_ids'][mini_batch_inds],
                'attention_mask': input_data['attention_mask'][mini_batch_inds]
            }
            mb_logits, mb_vpreds = model(**mb_model_inputs) # mb_logits 的维度是 (mini_batch_size, seq_len, vocab_size)，mb_vpreds 的维度是 (mini_batch_size, seq_len)
            mb_logits = torch.nn.functional.log_softmax(
                mb_logits[:, :-1, :],
                dim=-1
            ) #去掉最后一个token的logits，因为最后一个token是奖励token，不参与概率计算，然后对剩下的token的logits进行softmax沿着词汇表维度计算对数概率，得到每个token的log概率分布，维度为 (mini_batch_size, seq_len-1, vocab_size)
            mb_logprobs = torch.gather(
                mb_logits,
                2,
                mb_model_inputs['input_ids'][:, 1:].unsqueeze(-1)
            ).squeeze(-1)# mb_model_inputs['input_ids'][:, 1:] 是去掉第一个token的输入 ID，因为第一个token是输入的最后一个token，不参与概率计算，mb_logits 的维度是 (mini_batch_size, seq_len-1, vocab_size)，使用 gather 函数根据输入 ID 从 logit 中提取实际生成的 token 的 log 概率，得到 mb_logprobs 的维度为 (mini_batch_size, seq_len-1)

            loss, loss_v = compute_loss(
                logprobs[mini_batch_inds],
                values[mini_batch_inds],
                mb_logprobs,
                mb_vpreds[:, :-1],
                masks[mini_batch_inds],
                advantages[mini_batch_inds],
                returns[mini_batch_inds]
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            print('loss/total', loss.item())
    print('mini-batch training finished')

In [27]:
mini_batch_train()


loss/total 0.5331320762634277
loss/total 0.41266411542892456
loss/total 0.49528199434280396
loss/total 0.4474866986274719
loss/total 0.626681923866272
loss/total 0.3256409764289856
loss/total 0.4231930077075958
loss/total 0.7216978669166565
loss/total 0.38637882471084595
loss/total 0.45533251762390137
loss/total 0.48779815435409546
loss/total 0.15969030559062958
loss/total 0.31028375029563904
loss/total 0.3614738881587982
loss/total 0.35403990745544434
loss/total 0.24810917675495148
loss/total 0.34514757990837097
loss/total 0.13816851377487183
loss/total 0.25774285197257996
loss/total 0.35360196232795715
loss/total 0.16816756129264832
loss/total 0.6103640794754028
loss/total 0.3684444725513458
loss/total 0.21447692811489105
loss/total 0.2113909125328064
loss/total 0.25808027386665344
loss/total 0.414003849029541
loss/total 0.0816812738776207
loss/total 0.2054356336593628
loss/total 0.27023056149482727
loss/total 0.4838109612464905
loss/total 0.39708811044692993
mini-batch training fini

In [28]:
num_epochs = 1

for epoch in range(num_epochs):
    for batch in train_dataloader:
        # Generate responses
        query_tensors = batch['input_ids']
        query_attention_masks = batch['attention_mask']

        response_tensors = []
        query_response_tensors = []
        score_tensors = []

        for i, query in enumerate(query_tensors):
            query = query.to(device)
            query_attention_mask = query_attention_masks[i].to(device)
            new_tokens = random.choice(list(range(
                output_min_length,
                output_max_length)))
            generation_kwargs["max_new_tokens"] = new_tokens
            query_response = model.generate(
                input_ids=query.unsqueeze(0),
                attention_mask=query_attention_mask.unsqueeze(0),
                **generation_kwargs
            ).squeeze(0)

            response_len = len(query_response) - len(query)
            response_tensors.append(query_response[-response_len:])
            query_response_tensors.append(query_response)

            with torch.no_grad():
                query_response_score = torch.cat([
                    query_response,
                    torch.tensor([REWARD_TOKEN_ID]).to(device)])
                attention_mask = torch.ones_like(
                    query_response_score,
                    dtype=torch.long)
                score = reward_model(
                    query_response_score.unsqueeze(0),
                    attention_mask.unsqueeze(0)
                ).squeeze(0)[-1]
                score = 2 * (score - 0.5)
            score_tensors.append(score)

        input_data = data_collator([
            {
                'input_ids': ids,
                'attention_mask': torch.ones_like(ids)
            }
            for ids in query_response_tensors
        ]).to(device)

        # 奖励和优势
        logprobs, rewards, values, masks = compute_rewards(
            input_data,
            query_tensors,
            response_tensors,
            score_tensors
        )
        advantages, returns = compute_advantage(rewards, values, masks)

        # 小批次训练
        mini_batch_train()
    print(f'epoch {epoch + 1} finished')

loss/total 0.35303229093551636
loss/total 0.3235873281955719
loss/total 0.6674754619598389
loss/total 0.32710397243499756
loss/total 0.3884756863117218
loss/total 0.2383279800415039
loss/total 0.35746458172798157
loss/total 0.45238810777664185
loss/total 0.15627914667129517
loss/total 0.1981133222579956
loss/total 0.1337571144104004
loss/total 0.36142778396606445
loss/total 0.5334069132804871
loss/total 0.05265199393033981
loss/total 0.5146079063415527
loss/total 0.03354935720562935
loss/total 0.3435990810394287
loss/total 0.4584532082080841
loss/total 0.401809960603714
loss/total -0.05451885983347893
loss/total 0.04724056273698807
loss/total 0.3305615782737732
loss/total -0.21515585482120514
loss/total 0.2835950255393982
loss/total 0.3137166500091553
loss/total 0.215018630027771
loss/total -0.09878623485565186
loss/total 0.3022336959838867
loss/total -0.029757287353277206
loss/total 0.10715430974960327
loss/total 0.545037031173706
loss/total 0.21708308160305023
mini-batch training fin

KeyboardInterrupt: 

In [30]:
print(len(tokenized_dataset_val))
print(len(tokenized_dataset_train))
val_gen_lengths = [0] * len(tokenized_dataset_val)
for i in range(len(tokenized_dataset_val)):
    val_gen_lengths[i] = random.choice(list(range(
        output_min_length,
        output_max_length)))
val_gen_lengths[:10]


5600
19907


[6, 9, 15, 15, 19, 8, 17, 13, 7, 6]

In [35]:
def validate():
    scores = []
    for b, batch in enumerate(val_dataloader):
        if b > 10: # 只验证前10个批次，避免验证时间过长
            break
        # Generate_responses
        query_tensors = batch['input_ids']
        query_attention_masks = batch['attention_mask']
        for i, query in enumerate(query_tensors):
            query = query.to(device)
            query_attention_mask = query_attention_masks[i].to(device)
            new_tokens = val_gen_lengths[b * len(query_tensors) + i]
            generation_kwargs["max_new_tokens"] = new_tokens
            query_response = model.generate(
                input_ids=query.unsqueeze(0),
                attention_mask=query_attention_mask.unsqueeze(0),
                **generation_kwargs
            ).squeeze(0)
            query_response_score = torch.cat([
                query_response,
                torch.tensor([REWARD_TOKEN_ID]).to(device)])
            attention_mask = torch.ones_like(
                query_response_score, dtype=torch.long)
            score = reward_model(
                query_response_score.unsqueeze(0),
                attention_mask.unsqueeze(0)
            ).squeeze(0)[-1]
            score = 2 * (score - 0.5)
            scores.append(score.item())
    print('平均分数:', sum(scores) / len(scores))

validate()

平均分数: -0.6721530788662758


In [36]:
torch.save(model.state_dict(), 'gpt2-ppo.pt')

In [37]:
model_path = '../SFT微调大模型/models/fine_tuned_gpt2'
model = ModelForCausalLMWithValueHead(model_path).to(device)
validate()

平均分数: -0.6735233451155099
